# ACHTUNG: In diesem Notebook werden händisch Werte nachgetragen. Am besten Notebook nachvollziehen und mit dem nächsten Datensatz im nächsten Notebook weitermachen!

### Prüfen NAs und 0 Values der Spalte 'current_team' und 'team_class' 

In [2]:
import pandas as pd
import os


In [3]:
df = pd.read_pickle('../../data/processed/15_cleaned_master_data.pkl')

In [ ]:
# Prüfen ob die Indizes wieder gleich sind

indizes_uebereinstimmung = (df['current_team'].isnull() == df['team_class'].isnull()).all()

print(f"current_team und teamm_class selben Indizes? -> {indizes_uebereinstimmung}")


current_team und teamm_class selben Indizes? -> True


Analyse wie viel Fahrer es betrifft

In [16]:
# erneute Ausgabe csv zum Korrigieren
df_missing_team = df[df['current_team'].isnull()]


recherche_team_df = df_missing_team[['name', 'year', 'nationality', 'rider_url']].drop_duplicates().copy()


recherche_team_df['current_team_recherchiert'] = ""
recherche_team_df['team_class_recherchiert'] = ""


recherche_team_df = recherche_team_df.sort_values(by=['name', 'year'])


target_path = "../../data/interim/fahrer_team_recherche.csv"
recherche_team_df.to_csv(target_path, index=False, sep=';')

In [ ]:
# Hilfscode für die Recherche

df_euskadi = df[df['current_team'].str.contains('vict', case=False, na=False)]
euskadi_kombis = df_euskadi[['current_team', 'team_class']].drop_duplicates()
print(euskadi_kombis)


               current_team team_class
51690  Bahrain - Victorious         WT


In [15]:
# Erneute Ausgabe einer Korrigier CSV

df_missing_team = df[df['current_team'].isnull()]

df_to_correct = df_missing_team[['name', 'year', 'rider_url']].drop_duplicates().copy()

df_to_correct['corrected_team'] = ""
df_to_correct['corrected_team_class'] = ""

df_to_correct = df_to_correct.sort_values(by=['year', 'name'])

output_csv = '02_missing_teams_to_correct.csv'
df_to_correct.to_csv(output_csv, index=False, encoding='utf-8-sig')


### Merge

In [ ]:
recherche_team = pd.read_csv('../../data/interim/fahrer_team_recherche.csv', sep=';')


recherche_team['current_team_recherchiert'] = recherche_team['current_team_recherchiert'].astype(str).str.strip()
recherche_team['team_class_recherchiert'] = recherche_team['team_class_recherchiert'].astype(str).str.strip()


df = df.merge(
    recherche_team[['rider_url', 'year', 'current_team_recherchiert', 'team_class_recherchiert']],
    on=['rider_url', 'year'],
    how='left'
)


df['current_team'] = df['current_team'].combine_first(df['current_team_recherchiert'])
df['team_class'] = df['team_class'].combine_first(df['team_class_recherchiert'])

#temporären Merge-Spalten wieder löschen
df.drop(columns=['current_team_recherchiert', 'team_class_recherchiert'], inplace=True)




team_nans = df['current_team'].isna().sum()
class_nans = df['team_class'].isna().sum()

print(f"Verbleibende NaNs in 'current_team': {team_nans}")
print(f"Verbleibende NaNs in 'team_class':   {class_nans}")


Verbleibende NaNs in 'current_team': 0
Verbleibende NaNs in 'team_class':   0


### Checkpoint
Neues .pkl erstellen 16

In [90]:
pfad = '../../data/processed/16_cleaned_master_data.pkl'
df.to_pickle(pfad)

In [91]:
df.isna().sum()

race                                0
year                                0
url                                 0
rank                                0
rider_url                           0
time_gap                            0
birthdate                           0
height                              0
name                                0
nationality                         0
weight                              0
url_name                            0
departure                           0
arrival                             0
distance                            0
vertical_meters                     0
won_how                             0
avg_speed                           0
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                              0
climber                             0
hills                               0
stage_nr                            0
date        